# 11. Optimizer updates — AdamW, Prodigy, Muon, and K3 Per-Head Muon

The examples operate on tiny matrices, but they implement the optimizer state
and update path rather than displaying a single matrix operation.


In [ ]:
import math
import torch

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. SGD, momentum, AdamW


In [ ]:
weight = torch.tensor(
    [[1.0, -1.0], [0.5, 2.0]],
    device=device,
)
gradient = torch.tensor(
    [[0.2, -0.4], [1.0, 0.5]],
    device=device,
)

learning_rate = 0.1
momentum = torch.zeros_like(weight)

for step in range(3):
    momentum = 0.9 * momentum + gradient
    weight = weight - learning_rate * momentum

print("momentum SGD:", weight)

m = torch.zeros_like(weight)
v = torch.zeros_like(weight)

for step in range(1, 4):
    m = 0.9 * m + 0.1 * gradient
    v = 0.999 * v + 0.001 * gradient.square()

    m_hat = m / (1 - 0.9 ** step)
    v_hat = v / (1 - 0.999 ** step)

    weight = (
        weight * (1 - learning_rate * 0.01)
        - learning_rate * m_hat / (v_hat.sqrt() + 1e-8)
    )

print("AdamW:", weight)


## 2. Prodigy D-adaptation state


In [ ]:
class TinyProdigyState:
    def __init__(self, parameter):
        self.p0 = parameter.clone()
        self.s = torch.zeros_like(parameter)
        self.exp_avg = torch.zeros_like(parameter)
        self.exp_avg_sq = torch.zeros_like(parameter)
        self.d = 1e-3
        self.d_max = self.d
        self.d_numerator = 0.0


def prodigy_step(parameter, gradient, state, beta1=0.9, beta2=0.999):
    beta3 = math.sqrt(beta2)
    d0 = 1e-3
    adapted_lr = state.d

    displacement = state.p0 - parameter
    delta_numerator = (
        (state.d / d0)
        * adapted_lr
        * torch.sum(gradient * displacement).item()
    )
    state.d_numerator = (
        beta3 * state.d_numerator
        + delta_numerator
    )

    state.s = (
        beta3 * state.s
        + ((state.d / d0) * adapted_lr) * gradient
    )

    denominator = state.s.abs().sum().item()
    if denominator > 0:
        d_hat = state.d_numerator / denominator
        state.d_max = max(state.d_max, d_hat)
        state.d = max(state.d, state.d_max)

    state.exp_avg = (
        beta1 * state.exp_avg
        + state.d * (1 - beta1) * gradient
    )
    state.exp_avg_sq = (
        beta2 * state.exp_avg_sq
        + state.d**2 * (1 - beta2) * gradient.square()
    )

    update = state.exp_avg / (
        state.exp_avg_sq.sqrt() + state.d * 1e-8
    )
    return parameter - update


parameter = torch.tensor(
    [[1.0, -1.0], [0.5, 2.0]],
    device=device,
)
state = TinyProdigyState(parameter)

for scale in [1.0, 0.7, 0.4]:
    parameter = prodigy_step(
        parameter,
        scale * gradient,
        state,
    )
    print("d:", state.d)


## 3. Muon: momentum -> Nesterov -> Newton-Schulz -> scaling

The hidden matrix update is orthogonalized instead of normalized elementwise.


In [ ]:
def newton_schulz_quintic(matrix, steps=5):
    a = 3.4445
    b = -4.7750
    c = 2.0315

    x = matrix.float()
    transposed = x.size(-2) > x.size(-1)

    if transposed:
        x = x.mT

    x = x / (
        x.norm(dim=(-2, -1), keepdim=True)
        + 1e-7
    )

    for _ in range(steps):
        gram = x @ x.mT
        polynomial = b * gram + c * (gram @ gram)
        x = a * x + polynomial @ x

    if transposed:
        x = x.mT
    return x.to(matrix.dtype)


def muon_direction(
    gradient,
    momentum_buffer,
    beta=0.95,
    nesterov=True,
):
    momentum_buffer.mul_(beta).add_(
        gradient,
        alpha=1 - beta,
    )

    if nesterov:
        raw_update = beta * momentum_buffer + (1 - beta) * gradient
    else:
        raw_update = momentum_buffer

    orthogonal = newton_schulz_quintic(raw_update)
    aspect_scale = math.sqrt(
        max(1.0, gradient.size(-2) / gradient.size(-1))
    )
    return aspect_scale * orthogonal


matrix_gradient = torch.randn(16, 12, device=device)
matrix_momentum = torch.zeros_like(matrix_gradient)
muon_update = muon_direction(
    matrix_gradient,
    matrix_momentum,
)
print("Muon update:", muon_update.shape)


## 4. Per-Head Muon

K3 partitions the attention projection by head and applies Muon's matrix
orthogonalization independently to each head. That is materially different
from flattening all attention heads into one matrix.


In [ ]:
def per_head_muon_direction(
    gradient,
    momentum,
    num_heads,
    beta=0.95,
):
    out_features, in_features = gradient.shape
    assert out_features % num_heads == 0

    rows_per_head = out_features // num_heads
    updates = []

    for head_index in range(num_heads):
        start = head_index * rows_per_head
        stop = start + rows_per_head

        head_gradient = gradient[start:stop]
        head_momentum = momentum[start:stop]

        head_update = muon_direction(
            head_gradient,
            head_momentum,
            beta=beta,
        )
        updates.append(head_update)

    return torch.cat(updates, dim=0)


num_heads = 4
qkv_gradient = torch.randn(
    num_heads * 8,
    24,
    device=device,
)
qkv_momentum = torch.zeros_like(qkv_gradient)

global_update = muon_direction(
    qkv_gradient,
    torch.zeros_like(qkv_gradient),
)
per_head_update = per_head_muon_direction(
    qkv_gradient,
    qkv_momentum,
    num_heads,
)

for head_index in range(num_heads):
    rows = slice(head_index * 8, (head_index + 1) * 8)
    print(
        f"head {head_index}:",
        per_head_update[rows].norm().item(),
    )

print(
    "global vs per-head difference:",
    (global_update - per_head_update).norm().item(),
)


## 5. One explicit parameter step

This cell makes the optimizer semantics visible: the computed Muon direction
is applied to a weight matrix, while vector/scalar parameters would normally
remain on an AdamW-style path.


In [ ]:
weight = torch.randn(32, 24, device=device)
gradient = torch.randn_like(weight)
momentum = torch.zeros_like(weight)

direction = per_head_muon_direction(
    gradient,
    momentum,
    num_heads=4,
)
learning_rate = 0.02
weight_decay = 0.01

updated_weight = (
    weight * (1 - learning_rate * weight_decay)
    - learning_rate * direction
)

print("parameter delta norm:", (updated_weight - weight).norm().item())


## References and provenance

- **Prodigy**: D-adaptation state is updated from parameter displacement and
  accumulated gradient statistics.
- **Muon**: momentum/Nesterov matrix update followed by Newton-Schulz
  orthogonalization.
- **Kimi K3 Per-Head Muon**: attention projection matrices are partitioned by
  head and optimized independently instead of orthogonalizing one giant matrix.
